In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import random
import os

In [0]:
# Схемы для данных
employees_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("hire_date", DateType(), True),
    StructField("city", StringType(), True),
    StructField("performance_score", IntegerType(), True)
])

sales_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("emp_id", IntegerType(), True),
    StructField("product_category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("sale_date", TimestampType(), True),
    StructField("region", StringType(), True),
    StructField("discount_applied", BooleanType(), True),
    StructField("customer_id", StringType(), True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("country", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("loyalty_tier", StringType(), True)
])

print("Генерация данных employees...")
# Генерация данных для employees
departments = ["IT", "HR", "Finance", "Marketing", "Sales", "Operations", "Engineering", "Support"]
cities = ["New York", "London", "Tokyo", "Berlin", "Sydney", "Toronto", "Paris", "Singapore"]

employees_data = []
for i in range(1, 501):
    employees_data.append((
        i,
        f"Employee_{i}",
        random.choice(departments),
        random.randint(45000, 160000),
        datetime(2018, 1, 1) + timedelta(days=random.randint(0, 2190)),
        random.choice(cities),
        random.randint(1, 100)
    ))

employees_df = spark.createDataFrame(employees_data, employees_schema)
print(f"Employees создан: {employees_df.count()} строк")

print("Генерация данных customers...")
# Генерация данных для customers
segments = ["Enterprise", "SMB", "Startup", "Individual"]
tiers = ["Bronze", "Silver", "Gold", "Platinum"]
countries = ["USA", "UK", "Germany", "Japan", "Australia", "Canada", "France", "Brazil", "India", "China"]

customers_data = []
for i in range(1, 201):
    customers_data.append((
        f"CUST{5000 + i}",
        f"Customer_{i}",
        random.choice(segments),
        random.choice(countries),
        datetime(2020, 1, 1) + timedelta(days=random.randint(0, 1460)),
        random.choice(tiers)
    ))

customers_df = spark.createDataFrame(customers_data, customers_schema)
print(f"Customers создан: {customers_df.count()} строк")

print("Генерация данных sales...")
# Генерация данных для sales
products = ["Electronics", "Clothing", "Home", "Books", "Sports", "Beauty", "Food", "Toys"]
regions = ["North America", "Europe", "Asia", "South America", "Africa", "Oceania"]

sales_data = []
for i in range(1, 1001):
    # Генерируем amount
    amount = random.uniform(50.0, 5000.0)
    amount = float(f"{amount:.2f}")
    
    # Создаем datetime
    total_seconds = random.randint(0, 365*24*60*60)
    sale_datetime = datetime(2023, 1, 1) + timedelta(seconds=total_seconds)
    
    sales_data.append((
        f"TXN{10000 + i}",
        random.randint(1, 500),
        random.choice(products),
        amount,
        sale_datetime,
        random.choice(regions),
        random.choice([True, False]),
        f"CUST{5000 + random.randint(1, 200)}"
    ))

sales_df = spark.createDataFrame(sales_data, sales_schema)
print(f"Sales создан: {sales_df.count()} строк")

# Создаем временные представления
employees_df.createOrReplaceTempView("employees")
sales_df.createOrReplaceTempView("sales")
customers_df.createOrReplaceTempView("customers")

print("Временные представления созданы!")
print("Доступные представления: employees, sales, customers")

# Покажем немного данных для проверки
print("Пример данных employees:")
employees_df.show(5)

print("Пример данных sales:")
sales_df.show(5)

print("Пример данных customers:")
customers_df.show(5)

In [0]:
sql_query_1 = """
SELECT 
    department,
    COUNT(*) as employee_count,
    ROUND(AVG(salary), 2) as avg_salary,
    MAX(salary) as max_salary,
    MIN(salary) as min_salary
FROM employees
WHERE salary > 50000 AND performance_score > 70
GROUP BY department
HAVING COUNT(*) >= 10
ORDER BY avg_salary DESC
"""

spark_query_1 = employees_df.filter((col('salary') > 50000) & (col('performance_score') > 70))\
    .groupBy('department')\
    .agg(
        count('*').alias('employee_count'),
        round(avg('salary'), 2).alias('avg_salary'),
        max('salary').alias('max_salary'),
        min('salary').alias('min_salary')
    ).filter(col('employee_count') >= 10)\
    .orderBy(col('avg_salary').desc())

spark.sql(sql_query_1).show(1, truncate=False, vertical=True)
spark_query_1.show(1, truncate=False, vertical=True)

In [0]:
sql_query_2 = """
SELECT 
    e.emp_id,
    e.name,
    e.department,
    e.city,
    s.transaction_id,
    s.product_category,
    s.amount,
    DATE(s.sale_date) as sale_date,
    s.region

FROM employees e
INNER JOIN sales s ON e.emp_id = s.emp_id
WHERE s.amount > 1000
  AND e.department IN ('Sales', 'Marketing')
  AND s.discount_applied = TRUE
  AND YEAR(s.sale_date) = 2023
ORDER BY s.amount DESC
LIMIT 15
"""

spark_query_2 = employees_df.alias('e').join(sales_df.alias('s'), 'emp_id', 'inner')\
    .filter((col('s.amount') > 1000) & (col('e.department').isin(['Sales', 'Marketing'])) & (col('s.discount_applied') == 'TRUE') & (year(col('s.sale_date')) == 2023))\
    .select(
        'e.emp_id',
        'e.name',
        'e.department',
        'e.city',
        's.transaction_id',
        's.product_category',
        's.amount',
        to_date(col('s.sale_date')).alias('sale_date'),
        's.region'
    ).orderBy(col('s.amount').desc()).limit(15)

spark.sql(sql_query_2).show(1, truncate=False, vertical=True)
spark_query_2.show(1, truncate=False, vertical=True)

In [0]:
sql_query_3 = """
SELECT 
    customer_id,
    customer_name,
    segment,
    country,
    loyalty_tier,
    signup_date,
    DATEDIFF(CURRENT_DATE(), signup_date) as days_as_customer,
    CASE 
        WHEN DATEDIFF(CURRENT_DATE(), signup_date) > 1095 THEN 'Loyal (3+ years)'
        WHEN DATEDIFF(CURRENT_DATE(), signup_date) > 730 THEN 'Established (2-3 years)'
        WHEN DATEDIFF(CURRENT_DATE(), signup_date) > 365 THEN 'Growing (1-2 years)'
        ELSE 'New (<1 year)'
    END as customer_tenure_category,
    RANK() OVER (PARTITION BY country ORDER BY signup_date) as earliest_signup_rank,
    COUNT(*) OVER (PARTITION BY country, segment) as segment_count_in_country
FROM customers
WHERE loyalty_tier IN ('Gold', 'Platinum')
ORDER BY country, days_as_customer DESC
"""

spark_query_3 = customers_df.filter(col('loyalty_tier').isin(['Gold', 'Platinum']))\
    .select(
        'customer_id',
        'customer_name',
        'segment',
        'country',
        'loyalty_tier',
        'signup_date',
        datediff(current_date(), col('signup_date')).alias('days_as_customer'),
        when(col('days_as_customer') > 1095, 'Loyal (3+ years)')\
        .when(col('days_as_customer') > 730, 'Established (2-3 years)')\
        .when(col('days_as_customer') > 365, 'Growing (1-2 years)')\
        .otherwise('New (<1 year)').alias('customer_tenure_category'),
        rank().over(Window.partitionBy('country').orderBy('signup_date')).alias('earliest_signup_rank'),
        count('*').over(Window.partitionBy('country', 'segment')).alias('segment_count_in_country')
    ).orderBy('country', col('days_as_customer').desc())

spark.sql(sql_query_3).show(1, truncate=False, vertical=True)
spark_query_3.show(1, truncate=False, vertical=True)

In [0]:
sql_query_4 = """
SELECT 
    c.country,
    c.segment,
    c.loyalty_tier,
    e.department,
    s.product_category,
    COUNT(DISTINCT s.transaction_id) as total_transactions,
    SUM(s.amount) as total_revenue,
    ROUND(AVG(s.amount), 2) as avg_transaction_value,
    COUNT(DISTINCT e.emp_id) as unique_employees,
    COUNT(DISTINCT c.customer_id) as unique_customers,
    SUM(CASE WHEN s.discount_applied THEN 1 ELSE 0 END) as discounted_transactions,
    ROUND(SUM(CASE WHEN s.discount_applied THEN s.amount ELSE 0 END) * 100.0 / SUM(s.amount), 2) as discount_revenue_percentage
FROM customers c
JOIN sales s ON c.customer_id = s.customer_id
JOIN employees e ON s.emp_id = e.emp_id
WHERE YEAR(s.sale_date) = 2023
    AND c.country IN ('USA', 'UK', 'Germany', 'Japan')
GROUP BY c.country, c.segment, c.loyalty_tier, e.department, s.product_category
HAVING total_transactions >= 5
ORDER BY total_revenue DESC
"""

spark_query_4 = customers_df.alias('c').join(sales_df.alias('s'), 'customer_id', 'inner')\
    .join(employees_df.alias('e'), 'emp_id', 'inner')\
    .where((year(col('s.sale_date')) == 2023) & (col('c.country').isin(['USA', 'UK', 'Germany', 'Japan'])))\
    .groupBy('c.country', 'c.segment', 'c.loyalty_tier', 'e.department', 's.product_category')\
    .agg(
        countDistinct('s.transaction_id').alias('total_transactions'),
        sum('s.amount').alias('total_revenue'),
        round(avg('s.amount'),2).alias('avg_transaction_value'),
        countDistinct('e.emp_id').alias('unique_employees'),
        countDistinct('c.customer_id').alias('unique_customers'),
        sum(when(col('s.discount_applied') == 'TRUE',1).otherwise(0)).alias('discounted_transactions'),
        round(sum(when(col('s.discount_applied') == 'TRUE', col('s.amount')).otherwise(0)) * 100.0 / col('total_revenue'), 2).alias('discount_revenue_percentage')
    ).filter(col('total_transactions') >= 5)\
    .orderBy(col('total_revenue').desc())

spark.sql(sql_query_4).show(1, truncate=False, vertical=True)
spark_query_4.show(1, truncate=False, vertical=True)

In [0]:
sql_query_5 = """
WITH employee_stats AS (
    SELECT 
        e.emp_id,
        e.name,
        e.department,
        e.city,
        e.salary,
        e.performance_score,
        COUNT(s.transaction_id) as total_sales,
        SUM(s.amount) as total_revenue,
        AVG(s.amount) as avg_sale_amount,
        MAX(s.amount) as max_sale_amount
    FROM employees e
    LEFT JOIN sales s ON e.emp_id = s.emp_id
    GROUP BY e.emp_id, e.name, e.department, e.city, e.salary, e.performance_score
)
SELECT 
    emp_id,
    name,
    department,
    city,
    salary,
    performance_score,
    total_sales,
    total_revenue,
    ROUND(avg_sale_amount, 2) as avg_sale_amount,
    max_sale_amount,
    ROUND(total_revenue / NULLIF(salary, 0), 2) as revenue_to_salary_ratio,
    RANK() OVER (PARTITION BY department ORDER BY total_revenue DESC) as dept_revenue_rank,
    CASE 
        WHEN total_revenue > 100000 THEN 'Top Performer'
        WHEN total_revenue > 50000 THEN 'Good Performer'
        WHEN total_revenue > 0 THEN 'Average Performer'
        ELSE 'No Sales'
    END as performance_category
FROM employee_stats
WHERE total_sales > 0
ORDER BY department, total_revenue DESC
"""

emp_stts = employees_df.alias('e').join(sales_df.alias('s'), 'emp_id', 'left')\
    .groupBy('e.emp_id', 'e.name', 'e.department', 'e.city', 'e.salary', 'e.performance_score')\
    .agg(
        countDistinct('s.transaction_id').alias('total_sales'),
        sum('s.amount').alias('total_revenue'),
        avg('s.amount').alias('avg_sale_amount'),
        max('s.amount').alias('max_sale_amount'))
    
spark_query_5 = emp_stts.filter(col('total_sales') > 0)\
    .select(
        'emp_id',
        'name',
        'department',
        'city',
        'salary',
        'performance_score',
        'total_sales',
        'total_revenue',
        round(col('avg_sale_amount'), 2).alias('avg_sale_amount'),
        'max_sale_amount',
        round(col('total_revenue') / nullif(col('salary'), lit(0)), 2).alias('revenue_to_salary_ratio'),
        rank().over(Window.partitionBy('department').orderBy(col('total_revenue').desc())).alias('dept_revenue_rank'),
        when(col('total_revenue') > 100000, 'Top Performer')\
        .when(col('total_revenue') > 50000, 'Good Performer')\
        .when(col('total_revenue') > 0, 'Average Performer')\
        .otherwise('No Sales').alias('performance_category')
    ).orderBy('department', col('total_revenue').desc())

spark.sql(sql_query_5).show(5, truncate=False, vertical=True)
spark_query_5.show(3, truncate=False, vertical=True)